# 03 — Mie scattering image from Sage API JSON and Go

Run notebook 01 first. This notebook consumes its eight certified boundary
sequences, constructs the Mie coefficients, evaluates the complete 800×800
total and scattered fields, writes binary data and PNG files, and displays
both images inline.

The sphere is circular because the Binder bundle enforces the square
`full_800` profile. Select the **Go (gonb)** kernel and run all cells.


In [ ]:
import (
    "encoding/binary"
    "encoding/json"
    "image"
    "image/color"
    "image/png"
    "github.com/janpfeifer/gonb/gonbui"
    "fmt"
    "math"
    "math/cmplx"
    "os"
    "runtime"
    "sync"
    "time"
)

// One-frame notebook port of the equations in pp473-mie-scattering.
// The original uses big.Float after float64 sin/cos seeds.  This compact port
// keeps the same recurrence and small-argument strategy in complex128.
type MieConfig struct {
    Width, Height, MaxL int
    Lambda, Radius      float64
    RefractiveIndex     complex128
    Mu                  float64
}

func configForProfile(profile string) MieConfig {
    switch profile {
    case "comparison":
        return MieConfig{
            Width: 64, Height: 64, MaxL: 8,
            Lambda: 0.2, Radius: 0.75 * 0.2,
            RefractiveIndex: complex(1.2, 0.05), Mu: 1.0,
        }
    case "full_800":
        return MieConfig{
            Width: 800, Height: 800, MaxL: 42,
            Lambda: 0.2, Radius: 3.18 * 0.2,
            RefractiveIndex: complex(1.33, 0.0), Mu: 1.0,
        }
    default:
        panic("profile must be comparison or full_800")
    }
}

func iPow(n int) complex128 {
    switch ((n % 4) + 4) % 4 {
    case 0: return 1
    case 1: return 1i
    case 2: return -1
    default: return -1i
    }
}

// j_l(z), j_l'(z), l=0,...,L.
// Miller downward recurrence is stable when l is large compared with |z|.
func sphericalJ(L int, z complex128) ([]complex128, []complex128) {
    if z == 0 {
        panic("the even-sized grid should never evaluate sphericalJ at zero")
    }
    top := L + 32 + int(cmplx.Abs(z))
    work := make([]complex128, top+2)
    work[top] = 1
    for l := top; l >= 1; l-- {
        work[l-1] = complex(float64(2*l+1), 0)*work[l]/z - work[l+1]
        // Keep the temporary dominant solution away from complex128 overflow.
        if cmplx.Abs(work[l-1]) > 1e200 {
            for m := l - 1; m < len(work); m++ {
                work[m] *= 1e-200
            }
        }
    }
    scale := (cmplx.Sin(z) / z) / work[0]
    vals := make([]complex128, L+1)
    ders := make([]complex128, L+1)
    for l := 0; l <= L; l++ {
        vals[l] = work[l] * scale
    }
    if L > 0 {
        ders[0] = -vals[1]
    }
    for l := 1; l <= L; l++ {
        ders[l] = vals[l-1] - complex(float64(l+1), 0)*vals[l]/z
    }
    return vals, ders
}

// y_l(x), y_l'(x), l=0,...,L, for positive real x.
func sphericalY(L int, x float64) ([]complex128, []complex128) {
    vals := make([]complex128, L+1)
    ders := make([]complex128, L+1)
    vals[0] = complex(-math.Cos(x)/x, 0)
    ders[0] = complex(math.Sin(x)/x+math.Cos(x)/(x*x), 0)
    if L > 0 {
        vals[1] = complex(-math.Cos(x)/(x*x)-math.Sin(x)/x, 0)
        ders[1] = vals[0] - 2*vals[1]/complex(x, 0)
    }
    for l := 2; l <= L; l++ {
        vals[l] = complex(float64(2*l-1)/x, 0)*vals[l-1] - vals[l-2]
        ders[l] = vals[l-1] - complex(float64(l+1)/x, 0)*vals[l]
    }
    return vals, ders
}

func sphericalH1(L int, x float64) ([]complex128, []complex128) {
    j, jp := sphericalJ(L, complex(x, 0))
    y, yp := sphericalY(L, x)
    h, hp := make([]complex128, L+1), make([]complex128, L+1)
    for l := 0; l <= L; l++ {
        h[l], hp[l] = j[l]+1i*y[l], jp[l]+1i*yp[l]
    }
    return h, hp
}

func legendre(L int, x float64) ([]float64, []float64) {
    p, dp := make([]float64, L+1), make([]float64, L+1)
    p[0] = 1
    if L > 0 { p[1], dp[1] = x, 1 }
    for l := 2; l <= L; l++ {
        p[l] = float64(2*l-1)/float64(l)*x*p[l-1] - float64(l-1)/float64(l)*p[l-2]
        dp[l] = float64(l)*p[l-1] + x*dp[l-1]
    }
    return p, dp
}

func multipole(st, ct float64, kr complex128, alpha, beta, z, zp []complex128, L int) float64 {
    p, dp := legendre(L, ct)
    var rcom, tcom complex128
    for l := 1; l <= L; l++ {
        ll1 := float64(l*(l+1))
        coeff := iPow(l-1) * complex(float64(2*l+1)/ll1, 0)
        zover := z[l] / kr
        rr := complex(ll1*st*dp[l], 0) * beta[l] * zover
        g := zp[l] + zover
        h := ct*dp[l] - ll1*p[l]
        tt := 1i*alpha[l]*z[l]*complex(dp[l], 0) - beta[l]*g*complex(h, 0)
        rcom += coeff * rr
        tcom += coeff * tt
    }
    // The source renderer plots the square of this real x-polarized amplitude.
    return real(rcom)*st + real(tcom)*ct
}

type SageBoundaryData struct {
    Schema string
    MaxL, Width, Height, WorkingBits int
    Lambda, Radius, Mu float64
    RefractiveIndex [2]float64
    BoundaryArguments map[string][2]float64
    Sequences map[string][][2]float64
}

func pairComplex(pair [2]float64) complex128 { return complex(pair[0], pair[1]) }

func complexSequence(data SageBoundaryData, name string) []complex128 {
    pairs, ok := data.Sequences[name]
    if !ok || len(pairs) != data.MaxL+1 { panic("missing or incorrectly sized Sage sequence: " + name) }
    out := make([]complex128, len(pairs))
    for i, pair := range pairs { out[i] = pairComplex(pair) }
    return out
}

func readSageBoundaryJSON(filename string) SageBoundaryData {
    raw, err := os.ReadFile(filename)
    if err != nil { panic(err) }
    var data SageBoundaryData
    if err := json.Unmarshal(raw, &data); err != nil { panic(err) }
    if data.Schema != "sage-spherical-sequences-for-go-v1" { panic("unexpected Sage JSON schema") }
    if data.Width != 800 || data.Height != 800 || data.MaxL != 42 { panic("expected full_800 Sage JSON profile") }
    if len(data.Sequences) != 8 { panic("Sage JSON must contain eight sequences") }
    for _, name := range []string{"JX", "JXDerivative", "JMX", "JMXDerivative", "YX", "YXDerivative", "HX", "HXDerivative"} {
        _ = complexSequence(data, name)
    }
    return data
}

func configFromSage(data SageBoundaryData) MieConfig {
    return MieConfig{
        Width: data.Width, Height: data.Height, MaxL: data.MaxL,
        Lambda: data.Lambda, Radius: data.Radius,
        RefractiveIndex: pairComplex(data.RefractiveIndex), Mu: data.Mu,
    }
}

// Boundary values and derivatives come exclusively from the certified Sage API JSON.
func mieCoefficientsFromSage(cfg MieConfig, data SageBoundaryData) (as, bs, ai, bi []complex128) {
    L := cfg.MaxL
    X := pairComplex(data.BoundaryArguments["X"])
    MX := pairComplex(data.BoundaryArguments["MX"])
    J, Jd := complexSequence(data, "JX"), complexSequence(data, "JXDerivative")
    H, Hd := complexSequence(data, "HX"), complexSequence(data, "HXDerivative")
    N, Nd := complexSequence(data, "JMX"), complexSequence(data, "JMXDerivative")
    mu, n := complex(cfg.Mu, 0), cfg.RefractiveIndex
    if cmplx.Abs(MX-n*X) > 1e-12*math.Max(1, cmplx.Abs(MX)) { panic("MX != refractive index * X") }
    as, bs = make([]complex128, L+1), make([]complex128, L+1)
    ai, bi = make([]complex128, L+1), make([]complex128, L+1)
    for l := 1; l <= L; l++ {
        Jp := X*Jd[l] + J[l]
        Hp := X*Hd[l] + H[l]
        Np := MX*Nd[l] + N[l]
        denA := mu*Hp*N[l] - H[l]*Np
        denB := mu*H[l]*Np - n*n*Hp*N[l]
        as[l] = (J[l]*Np - mu*Jp*N[l]) / denA
        bs[l] = (n*n*Jp*N[l] - mu*J[l]*Np) / denB
        ai[l] = mu * (J[l]*Hp - Jp*H[l]) / denA
        bi[l] = n * mu * (Jp*H[l] - J[l]*Hp) / denB
    }
    return
}

type radialRecord struct {
    inside bool
    kr complex128
    j, jp, h, hp []complex128
}

type radialResult struct {
    key int64
    record radialRecord
}

func radiusKey(cfg MieConfig, x, y int) int64 {
    dx := int64(2*x - (cfg.Width - 1))
    dy := int64((cfg.Height - 1) - 2*y)
    w, h := int64(cfg.Width), int64(cfg.Height)
    return dx*dx*h*h + dy*dy*w*w
}

func radiusFromKey(cfg MieConfig, key int64) float64 {
    return math.Sqrt(float64(key)) / float64(cfg.Width*cfg.Height)
}

func radialRecordForKey(cfg MieConfig, key int64) radialRecord {
    radius := radiusFromKey(cfg, key)
    k := 2 * math.Pi / cfg.Lambda
    if radius < cfg.Radius {
        kr := cfg.RefractiveIndex * complex(k*radius, 0)
        j, jp := sphericalJ(cfg.MaxL, kr)
        return radialRecord{inside: true, kr: kr, j: j, jp: jp}
    }

    krReal := k * radius
    kr := complex(krReal, 0)
    j, jp := sphericalJ(cfg.MaxL, kr)
    y, yp := sphericalY(cfg.MaxL, krReal)
    h := make([]complex128, cfg.MaxL+1)
    hp := make([]complex128, cfg.MaxL+1)
    for l := 0; l <= cfg.MaxL; l++ {
        h[l] = j[l] + 1i*y[l]
        hp[l] = jp[l] + 1i*yp[l]
    }
    return radialRecord{inside: false, kr: kr, j: j, jp: jp, h: h, hp: hp}
}

func uniqueRadiusKeys(cfg MieConfig) []int64 {
    seen := make(map[int64]struct{})
    for x := cfg.Width / 2; x < cfg.Width; x++ {
        for y := 0; y < cfg.Height; y++ {
            seen[radiusKey(cfg, x, y)] = struct{}{}
        }
    }
    keys := make([]int64, 0, len(seen))
    for key := range seen {
        keys = append(keys, key)
    }
    return keys
}

func buildRadialCache(cfg MieConfig, workers int) map[int64]radialRecord {
    keys := uniqueRadiusKeys(cfg)
    if workers > len(keys) { workers = len(keys) }
    if workers < 1 { workers = 1 }
    fmt.Printf(
        "radial cache: %d unique radii for %d half-plane pixels (%.2fx reuse)\n",
        len(keys), cfg.Width*cfg.Height/2,
        float64(cfg.Width*cfg.Height/2)/float64(len(keys)),
    )

    jobs := make(chan int64)
    results := make(chan radialResult)
    var wg sync.WaitGroup
    wg.Add(workers)
    for worker := 0; worker < workers; worker++ {
        go func() {
            defer wg.Done()
            for key := range jobs {
                results <- radialResult{key: key, record: radialRecordForKey(cfg, key)}
            }
        }()
    }
    go func() {
        for _, key := range keys { jobs <- key }
        close(jobs)
    }()

    cache := make(map[int64]radialRecord, len(keys))
    for range keys {
        result := <-results
        cache[result.key] = result.record
    }
    wg.Wait()
    return cache
}

func computeFrame(cfg MieConfig, boundary SageBoundaryData) ([]float64, []float64) {
    started := time.Now()
    workers := runtime.NumCPU()
    as, bs, ai, bi := mieCoefficientsFromSage(cfg, boundary)
    ones := make([]complex128, cfg.MaxL+1)
    for l := 1; l <= cfg.MaxL; l++ { ones[l] = 1 }

    cacheStarted := time.Now()
    cache := buildRadialCache(cfg, workers)
    fmt.Printf("radial phase: %v\n", time.Since(cacheStarted))

    total := make([]float64, cfg.Width*cfg.Height)
    scattered := make([]float64, cfg.Width*cfg.Height)
    stepx, stepy := 2.0/float64(cfg.Width), 2.0/float64(cfg.Height)
    cx, cy := float64(cfg.Width-1)/2, float64(cfg.Height-1)/2

    fieldStarted := time.Now()
    columns := make(chan int)
    var wg sync.WaitGroup
    wg.Add(workers)
    for worker := 0; worker < workers; worker++ {
        go func() {
            defer wg.Done()
            for x := range columns {
                for y := 0; y < cfg.Height; y++ {
                    fx, fy := stepx*(float64(x)-cx), stepy*(cy-float64(y))
                    r := math.Hypot(fx, fy)
                    st, ct := math.Abs(fx/r), fy/r
                    record := cache[radiusKey(cfg, x, y)]
                    var totalValue, scatteredValue float64
                    if record.inside {
                        amp := multipole(st, ct, record.kr, ai, bi, record.j, record.jp, cfg.MaxL)
                        totalValue = amp * amp
                    } else {
                        scAmp := multipole(st, ct, record.kr, as, bs, record.h, record.hp, cfg.MaxL)
                        incAmp := multipole(st, ct, record.kr, ones, ones, record.j, record.jp, cfg.MaxL)
                        totalValue = (scAmp + incAmp) * (scAmp + incAmp)
                        scatteredValue = scAmp * scAmp
                    }
                    i1, i2 := y*cfg.Width+x, y*cfg.Width+(cfg.Width-1-x)
                    total[i1], total[i2] = totalValue, totalValue
                    scattered[i1], scattered[i2] = scatteredValue, scatteredValue
                }
            }
        }()
    }
    for x := cfg.Width / 2; x < cfg.Width; x++ { columns <- x }
    close(columns)
    wg.Wait()
    fmt.Printf("angular/field phase: %v\n", time.Since(fieldStarted))
    fmt.Printf("total compute time: %v\n", time.Since(started))
    return total, scattered
}

func writeFloat64LE(filename string, values []float64) error {
    f, err := os.Create(filename)
    if err != nil { return err }
    defer f.Close()
    return binary.Write(f, binary.LittleEndian, values)
}

func rangeOf(values []float64) (float64, float64) {
    lo, hi := math.Inf(1), math.Inf(-1)
    for _, v := range values {
        if v < lo { lo = v }
        if v > hi { hi = v }
    }
    return lo, hi
}


func clamp01(x float64) float64 {
    if x < 0 { return 0 }
    if x > 1 { return 1 }
    return x
}

func heatColor(x float64) color.RGBA {
    x = math.Sqrt(clamp01(x))
    r := uint8(255 * clamp01(1.5-math.Abs(4*x-3)))
    g := uint8(255 * clamp01(1.5-math.Abs(4*x-2)))
    b := uint8(255 * clamp01(1.5-math.Abs(4*x-1)))
    return color.RGBA{r, g, b, 255}
}

func renderField(filename string, values []float64, width, height int) (*image.RGBA, error) {
    lo, hi := rangeOf(values)
    img := image.NewRGBA(image.Rect(0, 0, width, height))
    spread := hi-lo
    if spread == 0 { spread = 1 }
    for y := 0; y < height; y++ {
        for x := 0; x < width; x++ {
            img.SetRGBA(x, y, heatColor((values[y*width+x]-lo)/spread))
        }
    }
    f, err := os.Create(filename)
    if err != nil { return nil, err }
    defer f.Close()
    if err := png.Encode(f, img); err != nil { return nil, err }
    return img, nil
}


In [ ]:
%%
boundary := readSageBoundaryJSON("sage-spherical-sequences-for-go.json")
cfg := configFromSage(boundary)
total, scattered := computeFrame(cfg, boundary)
if err := writeFloat64LE("sage-api-go-mie-total.data", total); err != nil { panic(err) }
if err := writeFloat64LE("sage-api-go-mie-scattered.data", scattered); err != nil { panic(err) }
totalImage, err := renderField("sage-api-go-mie-total.png", total, cfg.Width, cfg.Height)
if err != nil { panic(err) }
scatteredImage, err := renderField("sage-api-go-mie-scattered.png", scattered, cfg.Width, cfg.Height)
if err != nil { panic(err) }
tmin, tmax := rangeOf(total)
smin, smax := rangeOf(scattered)
fmt.Printf("Sage API JSON: %s; %d certified boundary values consumed\n", boundary.Schema, 8*(cfg.MaxL+1))
fmt.Printf("Mie image: %dx%d, Lmax=%d, R/lambda=%.2f, n=%v\n", cfg.Width, cfg.Height, cfg.MaxL, cfg.Radius/cfg.Lambda, cfg.RefractiveIndex)
fmt.Printf("total range:     [%.16e, %.16e]\n", tmin, tmax)
fmt.Printf("scattered range: [%.16e, %.16e]\n", smin, smax)
gonbui.DisplayMarkdown("## Total Mie field")
if err := gonbui.DisplayImage(totalImage); err != nil { panic(err) }
gonbui.DisplayMarkdown("## Scattered Mie field")
if err := gonbui.DisplayImage(scatteredImage); err != nil { panic(err) }
fmt.Println("SAGE API JSON / GO MIE RENDER: PASS")
